# PatchCore Review Notebook (WideResNet50-2 Multilayer UMAP Follow-up, x224)

This notebook is the curated reproducibility and review notebook for the saved `x224` WideResNet50-2 multilayer PatchCore UMAP follow-up.

Default behavior:
- load the checked-in sweep CSVs and per-variant artifacts
- recompute confusion matrices and defect analysis from the local `x224` metadata
- display and save the selected variant's review plots
- surface the saved embedding artifacts and checked-in UMAP visualization without retraining

Optional execution mode:
-
-

## Submission Context

- Dataset notebook: `data/dataset/x224/benchmark_50k_5pct/notebook.ipynb`
- Dataset config: `data/dataset/x224/benchmark_50k_5pct/data_config.toml`
- Experiment config: `experiments/anomaly_detection/patchcore/wideresnet50/x224/multilayer_umap/train_config.toml`
- Artifact root: `experiments/anomaly_detection/patchcore/wideresnet50/x224/multilayer_umap/artifacts/patchcore-wideresnet50-multilayer-umap`
- Default selected variant: `topk_mb50k_r005_x224`
- Mode: artifact-first review with optional cached variant rendering

## Imports and Paths

This cell resolves the repo root, defines the canonical paths used by the notebook, and exposes the run flags.

In [ ]:
from pathlib import Path
import json
import subprocess
import shutil
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display
cwd = Path.cwd().resolve()
candidate_roots = [cwd, *cwd.parents]
REPO_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        REPO_ROOT = candidate
        break
if REPO_ROOT is None:
    raise RuntimeError('Could not locate repo root containing src/wafer_defect and configs/')
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))
from wafer_defect.evaluation import summarize_threshold_metrics
ARTIFACT_ROOT = REPO_ROOT / 'experiments/anomaly_detection/patchcore/wideresnet50/x224/multilayer_umap/artifacts/patchcore-wideresnet50-multilayer-umap'
RESULTS_DIR = ARTIFACT_ROOT / 'results'
PLOTS_DIR = ARTIFACT_ROOT / 'plots'
METADATA_PATH = REPO_ROOT / 'data/processed/x224/wm811k/metadata_50k_5pct.csv'
RUNNER_PATH = REPO_ROOT / 'scripts/run_patchcore_wrn50_x224_umap.py'
DATA_CONFIG_PATH = REPO_ROOT / 'data/dataset/x224/benchmark_50k_5pct/data_config.toml'
TRAIN_CONFIG_PATH = REPO_ROOT / 'experiments/anomaly_detection/patchcore/wideresnet50/x224/multilayer_umap/train_config.toml'
RAW_PICKLE_PATH = REPO_ROOT / 'data/raw/LSWMD.pkl'
RETRAIN = False
RETRAIN_SKIP_UMAP = False
NUM_WORKERS = 4
SELECTED_VARIANT_NAME = 'topk_mb50k_r005_x224'
RENDER_ALL_SAVED_VARIANTS = True
VARIANTS_TO_RENDER = []
SHOW_LEGACY_UMAP_IMAGE = True


## Optional Full Rerun

In [ ]:
if RETRAIN:
    command = [sys.executable, '-u', str(RUNNER_PATH), '--raw-pickle', str(RAW_PICKLE_PATH), '--data-config', str(DATA_CONFIG_PATH), '--train-config', str(TRAIN_CONFIG_PATH), '--output-dir', str(ARTIFACT_ROOT), '--num-workers', str(NUM_WORKERS)]
    if RETRAIN_SKIP_UMAP:
        command.append('--skip-umap')
    print('Launching full rerun with the same runner used by the Modal app:')
    print(' '.join(command))
    subprocess.run(command, check=True, cwd=REPO_ROOT)
else:
    print('RETRAIN = False, using saved artifacts from the canonical artifact root.')


## Metadata and Sweep Loading

This cell loads the saved sweep table, summary, and local metadata used for defect-level analysis.

In [ ]:
metadata_ready = METADATA_PATH.exists()
if metadata_ready:
    metadata = pd.read_csv(METADATA_PATH)
    test_metadata = metadata[metadata['split'] == 'test'].reset_index(drop=True)
else:
    metadata = pd.DataFrame()
    test_metadata = pd.DataFrame()
    print(f'[WARNING] Benchmark metadata is missing: {METADATA_PATH}. Defect-analysis cells will be skipped.')
sweep_results_path = RESULTS_DIR / 'patchcore_sweep_results.csv'
sweep_summary_path = RESULTS_DIR / 'patchcore_sweep_summary.json'
review_ready = sweep_results_path.exists() and sweep_summary_path.exists()
if review_ready:
    sweep_results_df = pd.read_csv(sweep_results_path)
    sweep_summary = json.loads(sweep_summary_path.read_text(encoding='utf-8'))
    selected_variant_name = str(SELECTED_VARIANT_NAME or sweep_summary['best_variant']['name'])
else:
    sweep_results_df = pd.DataFrame()
    sweep_summary = {}
    selected_variant_name = None
    print('[WARNING] Cached sweep artifacts are missing. Variant-review cells will be skipped until the artifacts folder is restored.')
if metadata_ready:
    display(metadata['split'].value_counts().rename_axis('split').to_frame('count'))
if not sweep_results_df.empty:
    display(sweep_results_df)
print(f'Selected variant: {selected_variant_name}')


## Variant Loaders

These helpers normalize the reorganized artifact layout, recompute metrics from cached CSVs, and regenerate per-variant plots without retraining.

In [ ]:
def load_variant_outputs(variant_name: str) -> dict[str, object]:
    variant_root = ARTIFACT_ROOT / variant_name
    summary_path = variant_root / 'results' / 'summary.json'
    if not summary_path.exists():
        raise FileNotFoundError(f'Missing summary for {variant_name}: {summary_path}')
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    variant_eval_dir = variant_root / 'results' / 'evaluation'
    variant_umap_dir = variant_root / 'results' / 'umap'
    variant_plots_dir = variant_root / 'plots'
    variant_checkpoints_dir = variant_root / 'checkpoints'
    val_scores_df = pd.read_csv(variant_eval_dir / 'val_scores.csv')
    test_scores_df = pd.read_csv(variant_eval_dir / 'test_scores.csv')
    threshold_sweep_df = pd.read_csv(variant_eval_dir / 'threshold_sweep.csv')
    threshold = float(summary['threshold'])
    metrics = summarize_threshold_metrics(test_scores_df['is_anomaly'].to_numpy(), test_scores_df['score'].to_numpy(), threshold)
    best_sweep = threshold_sweep_df.sort_values('f1', ascending=False).iloc[0].to_dict()
    return {'summary': summary, 'threshold': threshold, 'val_scores_df': val_scores_df, 'test_scores_df': test_scores_df, 'threshold_sweep_df': threshold_sweep_df, 'metrics': metrics, 'best_sweep': best_sweep, 'variant_root': variant_root, 'variant_eval_dir': variant_eval_dir, 'variant_umap_dir': variant_umap_dir, 'variant_plots_dir': variant_plots_dir, 'variant_checkpoints_dir': variant_checkpoints_dir}

def compute_failure_tables(test_metadata: pd.DataFrame, test_scores_df: pd.DataFrame, threshold: float):
    analysis_df = test_metadata.copy()
    analysis_df['score'] = test_scores_df.reset_index(drop=True)['score']
    analysis_df['predicted_anomaly'] = (analysis_df['score'] > threshold).astype(int)
    analysis_df['error_type'] = 'tn'
    analysis_df.loc[(analysis_df['is_anomaly'] == 0) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'fp'
    analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 0), 'error_type'] = 'fn'
    analysis_df.loc[(analysis_df['is_anomaly'] == 1) & (analysis_df['predicted_anomaly'] == 1), 'error_type'] = 'tp'
    error_summary_df = analysis_df.groupby('error_type').agg(count=('error_type', 'size'), mean_score=('score', 'mean')).reindex(['tp', 'fn', 'fp', 'tn'])
    defect_recall_df = analysis_df[analysis_df['is_anomaly'] == 1].groupby('defect_type').agg(count=('defect_type', 'size'), detected=('predicted_anomaly', 'sum'), mean_score=('score', 'mean')).sort_values(['detected', 'count'], ascending=[False, False])
    defect_recall_df['recall'] = defect_recall_df['detected'] / defect_recall_df['count']
    return (analysis_df, error_summary_df, defect_recall_df)

def render_variant_artifacts(variant_name: str, payload: dict[str, object]) -> dict[str, str]:
    threshold = float(payload['threshold'])
    val_scores_df = payload['val_scores_df']
    test_scores_df = payload['test_scores_df']
    threshold_sweep_df = payload['threshold_sweep_df']
    metrics = payload['metrics']
    best_sweep = payload['best_sweep']
    variant_plots_dir = payload['variant_plots_dir']
    variant_eval_dir = payload['variant_eval_dir']
    variant_plots_dir.mkdir(exist_ok=True)
    variant_eval_dir.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(val_scores_df['score'], bins=30, alpha=0.85, color='#577590')
    axes[0].axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
    axes[0].set_title(f'Validation Normal Score Distribution\n{variant_name}')
    axes[0].legend()
    axes[1].hist(test_scores_df[test_scores_df['is_anomaly'] == 0]['score'], bins=30, alpha=0.7, label='normal', color='#4d908e')
    axes[1].hist(test_scores_df[test_scores_df['is_anomaly'] == 1]['score'], bins=30, alpha=0.7, label='anomaly', color='#f3722c')
    axes[1].axvline(threshold, color='red', linestyle='--', label=f'threshold={threshold:.4f}')
    axes[1].set_title(f'Test Score Distribution\n{variant_name}')
    axes[1].legend()
    plt.tight_layout()
    fig.savefig(variant_plots_dir / 'score_distribution.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['precision'], label='precision')
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['recall'], label='recall')
    ax.plot(threshold_sweep_df['threshold'], threshold_sweep_df['f1'], label='f1')
    ax.axvline(threshold, color='red', linestyle='--', label=f'validation threshold = {threshold:.4f}')
    ax.axvline(best_sweep['threshold'], color='green', linestyle=':', label=f"best sweep threshold = {best_sweep['threshold']:.4f}")
    ax.set_title(f'Threshold Sweep on Test Split\n{variant_name}')
    ax.legend()
    plt.tight_layout()
    fig.savefig(variant_plots_dir / 'threshold_sweep.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    cm_array = np.asarray(metrics['confusion_matrix'], dtype=float)
    fig, ax = plt.subplots(figsize=(5, 4))
    im = ax.imshow(cm_array, cmap='Blues')
    ax.set_xticks([0, 1], labels=['pred_normal', 'pred_anomaly'])
    ax.set_yticks([0, 1], labels=['true_normal', 'true_anomaly'])
    ax.set_title(f'Confusion Matrix\n{variant_name}')
    for row_idx in range(cm_array.shape[0]):
        for col_idx in range(cm_array.shape[1]):
            ax.text(col_idx, row_idx, int(cm_array[row_idx, col_idx]), ha='center', va='center', color='black')
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    fig.savefig(variant_plots_dir / 'confusion_matrix.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    analysis_df, error_summary_df, defect_recall_df = compute_failure_tables(test_metadata, test_scores_df, threshold)
    analysis_df.to_csv(variant_eval_dir / 'analysis_with_predictions.csv', index=False)
    error_summary_df.reset_index().to_csv(variant_eval_dir / 'error_summary.csv', index=False)
    defect_recall_df.reset_index().to_csv(variant_eval_dir / 'defect_recall.csv', index=False)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].bar(error_summary_df.index.astype(str), error_summary_df['count'], color='#e76f51')
    axes[0].set_title(f'Prediction Outcome Counts\n{variant_name}')
    axes[0].set_ylabel('count')
    top_defects_df = defect_recall_df.head(10).reset_index()
    axes[1].barh(top_defects_df['defect_type'], top_defects_df['recall'], color='#8ab17d')
    axes[1].set_xlim(0.0, 1.0)
    axes[1].set_title('Top Defect-Type Recall')
    axes[1].set_xlabel('recall')
    axes[1].invert_yaxis()
    plt.tight_layout()
    fig.savefig(variant_plots_dir / 'defect_breakdown.png', dpi=200, bbox_inches='tight')
    plt.close(fig)
    return {'plots_dir': str(variant_plots_dir), 'evaluation_dir': str(variant_eval_dir)}


## Selected Variant Review

This cell loads the selected variant, shows the key metrics, and regenerates the main branch-level plots.

In [ ]:
if not review_ready or not selected_variant_name:
    selected_variant = {}
    summary = {}
    val_scores_df = pd.DataFrame()
    test_scores_df = pd.DataFrame()
    threshold_sweep_df = pd.DataFrame()
    metrics = {}
    best_sweep = {}
    threshold = float('nan')
    print('[WARNING] Selected-variant review is unavailable because the cached sweep artifacts are missing.')
else:
    selected_variant = load_variant_outputs(selected_variant_name)
    summary = selected_variant['summary']
    val_scores_df = selected_variant['val_scores_df']
    test_scores_df = selected_variant['test_scores_df']
    threshold_sweep_df = selected_variant['threshold_sweep_df']
    metrics = selected_variant['metrics']
    best_sweep = selected_variant['best_sweep']
    threshold = float(summary['threshold'])
    metrics_df = pd.DataFrame([{'metric': 'precision', 'value': metrics['precision']}, {'metric': 'recall', 'value': metrics['recall']}, {'metric': 'f1', 'value': metrics['f1']}, {'metric': 'auroc', 'value': metrics['auroc']}, {'metric': 'auprc', 'value': metrics['auprc']}, {'metric': 'threshold', 'value': threshold}])
    confusion_df = pd.DataFrame(metrics['confusion_matrix'], index=['true_normal', 'true_anomaly'], columns=['pred_normal', 'pred_anomaly'])
    display(metrics_df)
    display(confusion_df)
    plot_df = sweep_results_df.copy().sort_values(['f1', 'auroc'], ascending=False).reset_index(drop=True)
    plot_df['label'] = plot_df['name']
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].barh(plot_df['label'], plot_df['f1'], color='#355070')
    axes[0].set_title('PatchCore Variant F1')
    axes[0].invert_yaxis()
    axes[1].barh(plot_df['label'], plot_df['auroc'], color='#6d597a')
    axes[1].set_title('PatchCore Variant AUROC')
    axes[1].invert_yaxis()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'variant_comparison_metrics.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)


## Failure Analysis

This cell computes the selected variant's error breakdown and saves the summary CSVs used in the report.

In [ ]:
if review_ready and metadata_ready and not test_scores_df.empty and len(test_metadata) == len(test_scores_df):
    analysis_df, error_summary_df, defect_recall_df = compute_failure_tables(test_metadata, test_scores_df, threshold)
    analysis_df.to_csv(RESULTS_DIR / 'selected_analysis_with_predictions.csv', index=False)
    error_summary_df.reset_index().to_csv(RESULTS_DIR / 'selected_error_summary.csv', index=False)
    defect_recall_df.reset_index().to_csv(RESULTS_DIR / 'selected_defect_recall.csv', index=False)
    display(error_summary_df)
    display(defect_recall_df)
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    axes[0].bar(error_summary_df.index.astype(str), error_summary_df['count'], color=VARIANT_COLOR_ANOMALY if 'VARIANT_COLOR_ANOMALY' in globals() else '#e76f51')
    axes[0].set_title(f'Prediction Outcome Counts\n{selected_variant_name}')
    axes[0].set_ylabel('count')
    top_defects_df = defect_recall_df.head(10).reset_index()
    axes[1].barh(top_defects_df['defect_type'], top_defects_df['recall'], color=VARIANT_COLOR_DEFECT if 'VARIANT_COLOR_DEFECT' in globals() else '#8ab17d')
    axes[1].set_xlim(0.0, 1.0)
    axes[1].set_title('Top Defect-Type Recall')
    axes[1].set_xlabel('recall')
    axes[1].invert_yaxis()
    plt.tight_layout()
    fig.savefig(PLOTS_DIR / 'defect_breakdown.png', dpi=200, bbox_inches='tight')
    plt.show()
    plt.close(fig)
else:
    analysis_df = pd.DataFrame()
    error_summary_df = pd.DataFrame()
    defect_recall_df = pd.DataFrame()
    print('[WARNING] Test metadata is unavailable or misaligned with cached scores. Skipping defect-analysis plots for this cell.')

if sweep_results_df.empty and not selected_variant_name:
    rendered_variants_df = pd.DataFrame(columns=['variant_name', 'plots_dir', 'evaluation_dir'])
    print('[WARNING] Cached variant rendering is unavailable because no sweep artifacts were found.')
else:
    variant_names = sweep_results_df['name'].astype(str).tolist() if ('RENDER_ALL_CACHED_VARIANTS' in globals() and RENDER_ALL_CACHED_VARIANTS) or ('RENDER_ALL_SAVED_VARIANTS' in globals() and RENDER_ALL_SAVED_VARIANTS) else []
    if 'VARIANTS_TO_RENDER' in globals():
        variant_names.extend([str(name) for name in VARIANTS_TO_RENDER])
    if selected_variant_name:
        variant_names.append(selected_variant_name)
    ordered_variant_names = []
    seen = set()
    for name in variant_names:
        if name not in seen:
            ordered_variant_names.append(name)
            seen.add(name)
    rendered_rows = []
    for variant_name in ordered_variant_names:
        payload = load_variant_outputs(variant_name)
        render_info = render_variant_artifacts(variant_name, payload)
        rendered_rows.append({'variant_name': variant_name, 'plots_dir': render_info['plots_dir'], 'evaluation_dir': render_info['evaluation_dir']})
    rendered_variants_df = pd.DataFrame(rendered_rows)
display(rendered_variants_df)


## UMAP Assets

This cell inventories the saved embedding arrays and displays the checked-in UMAP plot for the selected variant. It keeps the branch runnable without requiring a fresh embedding extraction pass.

In [ ]:
REGENERATE_UMAP = False
if not selected_variant or 'variant_umap_dir' not in selected_variant or 'variant_plots_dir' not in selected_variant:
    selected_umap_dir = None
    selected_plots_dir = None
    umap_png_path = None
    print('[WARNING] Selected-variant UMAP assets are unavailable because no cached variant could be restored.')
else:
    selected_umap_dir = selected_variant['variant_umap_dir']
    selected_plots_dir = selected_variant['variant_plots_dir']
    umap_png_path = selected_plots_dir / 'umap_by_split.png'
    if REGENERATE_UMAP:
        try:
            import umap as umap_module
            from wafer_defect.evaluation.umap_reference import export_reference_umap_bundle
            required_npy = {'train_embeddings': selected_umap_dir / 'train_embeddings.npy', 'val_embeddings': selected_umap_dir / 'val_embeddings.npy', 'val_labels': selected_umap_dir / 'val_labels.npy', 'test_embeddings': selected_umap_dir / 'test_embeddings.npy', 'test_labels': selected_umap_dir / 'test_labels.npy'}
            missing = [k for k, p in required_npy.items() if not p.exists()]
            if missing:
                raise FileNotFoundError(f'Missing embedding files (re-run training to regenerate): {missing}')
            train_emb = np.load(required_npy['train_embeddings'])
            val_emb = np.load(required_npy['val_embeddings'])
            val_lbl = np.load(required_npy['val_labels'])
            test_emb = np.load(required_npy['test_embeddings'])
            test_lbl = np.load(required_npy['test_labels'])
            umap_out = export_reference_umap_bundle(output_dir=selected_umap_dir, umap_module=umap_module, train_normal_embeddings=train_emb, val_embeddings=val_emb, val_labels=val_lbl, test_embeddings=test_emb, test_labels=test_lbl, pca_components=50, n_neighbors=15, min_dist=0.1, knn_k=15, metric='euclidean', random_state=42)
            generated_png = selected_umap_dir / 'plots' / 'embedding_umap.png'
            if generated_png.exists():
                shutil.copy2(generated_png, umap_png_path)
            print(f'UMAP regenerated and saved to {umap_png_path}')
        except ImportError:
            print('umap-learn not available. Install with: pip install umap-learn')
        except FileNotFoundError as e:
            print(f'Cannot regenerate UMAP: {e}')
    if umap_png_path.exists():
        shutil.copy2(umap_png_path, PLOTS_DIR / 'selected_variant_umap_by_split.png')
        display(Image(filename=str(umap_png_path)))
    else:
        print(f'UMAP image not found at {umap_png_path}')
        print('Set REGENERATE_UMAP = True to compute it (requires embedding .npy files).')


## Cached Variant Rendering

This section regenerates the standardized evaluation plots for each saved variant from cached CSVs without retraining.

In [ ]:
if sweep_results_df.empty and not selected_variant_name:
    rendered_variants_df = pd.DataFrame(columns=['variant_name', 'plots_dir', 'evaluation_dir', 'checkpoint_present'])
    print('[WARNING] Cached variant rendering is unavailable because no sweep artifacts were found.')
else:
    variant_names = sweep_results_df['name'].astype(str).tolist() if RENDER_ALL_SAVED_VARIANTS else []
    variant_names.extend([str(name) for name in VARIANTS_TO_RENDER])
    if selected_variant_name:
        variant_names.append(selected_variant_name)
    ordered_variant_names = []
    seen = set()
    for name in variant_names:
        if name not in seen:
            ordered_variant_names.append(name)
            seen.add(name)
    rendered_rows = []
    for variant_name in ordered_variant_names:
        payload = load_variant_outputs(variant_name)
        render_info = render_variant_artifacts(variant_name, payload)
        rendered_rows.append({'variant_name': variant_name, 'plots_dir': render_info['plots_dir'], 'evaluation_dir': render_info['evaluation_dir'], 'checkpoint_present': (payload['variant_checkpoints_dir'] / 'best_model.pt').exists()})
    rendered_variants_df = pd.DataFrame(rendered_rows)
display(rendered_variants_df)


## Saved Outputs

This cell summarizes the branch-level outputs created or refreshed by the review notebook.

In [ ]:
selected_variant_checkpoint = ''
if selected_variant and 'variant_checkpoints_dir' in selected_variant:
    selected_variant_checkpoint = str(selected_variant['variant_checkpoints_dir'] / 'best_model.pt')
saved_outputs = {'artifact_root': str(ARTIFACT_ROOT), 'results_dir': str(RESULTS_DIR), 'plots_dir': str(PLOTS_DIR), 'selected_variant_name': selected_variant_name, 'selected_variant_checkpoint': selected_variant_checkpoint, 'rendered_variants': rendered_variants_df['variant_name'].tolist() if 'variant_name' in rendered_variants_df.columns else []}
saved_outputs
